# SentinelPay: Model Competition and Evaluation
## Notebook 04 — Logistic Regression vs Random Forest vs XGBoost

**Author:** SentinelPay Research Team  
**Objective:** Compare three algorithmic families on the untouched holdout test set using metrics appropriate for extreme class imbalance. Select a champion model for production deployment.

---

### 1. Evaluation Methodology

Under extreme class imbalance, **accuracy is misleading**. A model predicting all transactions as legitimate achieves ~98.8% accuracy but catches zero fraud.

**Primary Metrics:**
- **PR-AUC (Precision-Recall Area Under Curve):** The gold standard for imbalanced classification. A random classifier achieves PR-AUC equal to the fraud prevalence (~0.012); a perfect classifier achieves 1.0.
- **Recall (Sensitivity):** Fraction of actual fraud transactions correctly detected. In fraud detection, missed fraud (False Negatives) directly translates to financial loss.
- **Precision:** Fraction of flagged transactions that are actually fraudulent. Low precision creates operational burden from investigating false alarms.
- **F1-Score:** Harmonic mean of Precision and Recall.

**Secondary Metrics:**
- **ROC-AUC:** Area under the Receiver Operating Characteristic curve.
- **Confusion Matrix:** Raw counts of TP, FP, TN, FN.

In [ ]:
import json
import pandas as pd
import numpy as np

# Load pre-computed evaluation results from training pipeline
with open('../models/model_comparison_metrics.json', 'r') as f:
    metrics = json.load(f)

print(f"Holdout Test Set: 12,000 transactions (untouched during training)")
print(f"Fraud Prevalence in Test: ~1.20%")

### 2. Head-to-Head Model Comparison

In [ ]:
rows = []
for name, m in metrics['candidate_models'].items():
    rows.append({
        'Model': name,
        'Precision': f"{m['precision']*100:.2f}%",
        'Recall': f"{m['recall']*100:.2f}%",
        'F1-Score': f"{m['f1_score']:.4f}",
        'ROC-AUC': f"{m['roc_auc']:.4f}",
        'PR-AUC': f"{m['pr_auc']:.4f}"
    })

df_comp = pd.DataFrame(rows).set_index('Model')
print("Model Comparison on Holdout Test Set (N=12,000):")
print("=" * 80)
df_comp

### 3. Confusion Matrix Analysis

In [ ]:
print("Confusion Matrices (Holdout Test Set):")
print("=" * 70)
for name, m in metrics['candidate_models'].items():
    cm = m['confusion_matrix']
    tn, fp = cm[0]
    fn, tp = cm[1]
    print(f"\n{name}:")
    print(f"                    Predicted Legit   Predicted Fraud")
    print(f"  Actual Legit      TN = {tn:>6,}       FP = {fp:>4}")
    print(f"  Actual Fraud      FN = {fn:>6}        TP = {tp:>4}")
    fpr = fp / (fp + tn) * 100
    fnr = fn / (fn + tp) * 100
    print(f"  False Positive Rate: {fpr:.3f}%  |  False Negative Rate: {fnr:.2f}%")

### 4. Champion Selection Rationale

**XGBoost is selected as the champion model** based on:

| Criterion | XGBoost Advantage |
|:----------|:------------------|
| **PR-AUC** | 0.9990 (highest among all candidates) |
| **Precision** | 97.93% (only 3 false positives on 12,000 transactions) |
| **Recall** | 98.61% (missed only 2 fraud cases) |
| **False Positive Rate** | 0.03% (minimal operational burden) |
| **SHAP Compatibility** | TreeSHAP provides exact, sub-millisecond local explanations |

While Random Forest also performs well, XGBoost achieves strictly superior precision (97.93% vs 94.67%) with identical recall, reducing false alarms by 62.5%.

Logistic Regression, despite strong recall, suffers from 56.85% precision (104 false positives), making it unsuitable for production where each false alarm triggers an expensive manual review.

In [ ]:
champion = metrics['champion']
cm = champion['metrics']['confusion_matrix']
tn, fp = cm[0]
fn, tp = cm[1]

print("CHAMPION MODEL: XGBoost")
print("=" * 50)
print(f"  Precision:  {champion['metrics']['precision']*100:.2f}%")
print(f"  Recall:     {champion['metrics']['recall']*100:.2f}%")
print(f"  F1-Score:   {champion['metrics']['f1_score']:.4f}")
print(f"  PR-AUC:     {champion['metrics']['pr_auc']:.4f}")
print(f"  ROC-AUC:    {champion['metrics']['roc_auc']:.4f}")
print(f"\n  True Positives:  {tp:>4}  (Fraud caught)")
print(f"  False Negatives: {fn:>4}  (Fraud missed)")
print(f"  False Positives: {fp:>4}  (Legit falsely flagged)")
print(f"  True Negatives:  {tn:>4}  (Legit correctly cleared)")

---
*Proceed to Notebook 05: Threshold Calibration and Operational Action Tiers.*